# AI Scripts Training with Logging
## Topics Covered:
1. **AI Scripts** - Production-ready patterns
2. **Logging** - Comprehensive logging system
3. **OpenAI** - API integration
4. **Batch Processing** - Efficient API calls
5. **Error Handling** - Retry logic and fallbacks


# 🧠 Building AI Scripts with Python
## Batch Processing · Chaining · Logging

---

Welcome, future AI Engineers! This notebook is your **hands-on guide** to writing production-ready AI scripts.

### What You'll Learn

| Topic | Why It Matters |
|-------|----------------|
| **Batch Processing** | Send hundreds of AI requests efficiently using concurrency and rate-limiting |
| **Chaining** | Connect multiple AI calls into a pipeline (output of one → input of next) |
| **Logging** | Track every AI call — tokens, duration, errors — for debugging and cost control |

### The Three Pillars of Production AI Code

```
┌─────────────────────────────────────────────────────────────────┐
│                  PRODUCTION AI SCRIPT                           │
│                                                                 │
│   ┌──────────────┐    ┌──────────────┐    ┌──────────────┐     │
│   │   BATCH      │───▶│   CHAIN      │───▶│    LOG       │     │
│   │  Processing  │    │   (Pipeline) │    │  (Observability)│   │
│   └──────────────┘    └──────────────┘    └──────────────┘     │
│         │                    │                    │             │
│         │  Speed            │  Accuracy           │  Trust      │
│         ▼                    ▼                    ▼             │
│   Handle volume          Break complexity      Know what       │
│   efficiently            into clear steps      happened        │
└─────────────────────────────────────────────────────────────────┘
```

---

**Target Audience:** Beginners to intermediate Python developers who want to build real AI-powered applications.

**Prerequisites:** Basic Python (functions, classes, lists, dicts). No prior AI/LLM experience needed.

---
# ⚙️ Setup & Installation

Let's get our environment ready. We need:
1. **OpenAI Python SDK** — to talk to the AI models
2. **python-dotenv** — to load API keys from a `.env` file
3. **An API Key** — from [platform.openai.com/api-keys](https://platform.openai.com/api-keys)

> 💡 **No API Key? No problem!** This notebook includes a **DEMO MODE** — when no key is detected, it simulates AI responses so you can still learn the patterns.

In [1]:
# ─────────────────────────────────────────────────────────────
#  STEP 1: Install required Python packages
# ─────────────────────────────────────────────────────────────
!pip install openai python-dotenv --quiet

print("✅ Packages installed successfully!")

✅ Packages installed successfully!


In [2]:
# ─────────────────────────────────────────────────────────────
#  STEP 2: Set up your API key
# ─────────────────────────────────────────────────────────────
# Option A: Set it directly (for quick testing)
# import os
# os.environ['OPENAI_API_KEY'] = 'sk-your-key-here'

# Option B: Load from .env file (RECOMMENDED for real projects)
from dotenv import load_dotenv
import os

load_dotenv()  # Looks for a .env file in the current directory

# Option C: Use Google Colab's secrets (uncomment if in Colab)
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# ── Check if key is set ─────────────────────────────────────
if os.getenv('OPENAI_API_KEY'):
    print(f"✅ API key found: {os.getenv('OPENAI_API_KEY')[:8]}...{os.getenv('OPENAI_API_KEY')[-4:]}")
    print(f"   Model: gpt-4o-mini (or whichever you prefer)")
else:
    print("⚠️  No API key detected — running in DEMO MODE")
    print("   The notebook will simulate AI responses.")
    print("   To use real AI, add your key above and re-run this cell.")

⚠️  No API key detected — running in DEMO MODE
   The notebook will simulate AI responses.
   To use real AI, add your key above and re-run this cell.


In [3]:
# ─────────────────────────────────────────────────────────────
#  STEP 3: Import all libraries we'll use
# ─────────────────────────────────────────────────────────────
import json
import time
import asyncio
import logging
import csv
import random
from datetime import datetime
from pathlib import Path
from typing import Optional, List, Dict, Any

print(f"✅ All imports loaded at {datetime.now().strftime('%H:%M:%S')}")
print(f"   Python version: {__import__('sys').version}")

✅ All imports loaded at 22:42:44
   Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


---
# 🏗️ AI Client Setup

We create a **lazy-loaded client** — it only connects when we first use it. This lets the notebook load without errors even when no API key is set.

> **Why lazy loading?** So you can import functions, explore the code, and even run demo mode without needing a key. The client 'wakes up' only when you make an actual AI call.

In [4]:
# ─────────────────────────────────────────────────────────────
#  AI CLIENT with DEMO MODE fallback
# ─────────────────────────────────────────────────────────────

DEMO_MODE = not os.getenv('OPENAI_API_KEY')

if not DEMO_MODE:
    from openai import OpenAI, AsyncOpenAI, APIError, RateLimitError, APITimeoutError
    DEFAULT_MODEL = "gpt-4o-mini"
    print("✅ Real AI mode — using OpenAI API")
else:
    DEFAULT_MODEL = "demo"
    print("🎭 Demo mode — responses will be simulated")

# Lazy-loaded client instances
_client_instance = None
_async_client_instance = None

def get_client():
    """Return sync client (creates on first call)."""
    global _client_instance
    if DEMO_MODE:
        return None
    if _client_instance is None:
        _client_instance = OpenAI()
    return _client_instance

def get_async_client():
    """Return async client (creates on first call)."""
    global _async_client_instance
    if DEMO_MODE:
        return None
    if _async_client_instance is None:
        _async_client_instance = AsyncOpen
        \AI()
    return _async_client_instance

print("✅ AI Client ready")

🎭 Demo mode — responses will be simulated
✅ AI Client ready


---
---

# 📝 PART 1: LOGGING — Know What Your AI Script Did

## Why Logging Matters

Imagine running an AI script that processes 1,000 customer reviews. Without logging:
- ❌ You don't know if it finished successfully
- ❌ You can't track how many tokens you used ($$$!)
- ❌ Errors are silent — you don't know which reviews failed
- ❌ You can't audit or debug individual results

**With logging:**
- ✅ Every single AI call is recorded with timestamp, tokens, duration
- ✅ Errors are captured and grouped
- ✅ You can export logs to CSV for analysis
- ✅ You have a complete audit trail

## What We'll Build

An `AILogger` class that:
1. Writes structured **JSONL** logs (one JSON object per line)
2. Prints a **console summary** for real-time monitoring
3. Tracks **tokens, duration, errors, chain steps, and batch IDs**
4. Can **export to CSV** for spreadsheet analysis

```jsonl
{"timestamp": "2026-06-15T10:30:00", "model": "gpt-4o-mini",
 "chain_step": "translate", "batch_id": "batch-001",
 "prompt_tokens": 45, "completion_tokens": 120,
 "total_tokens": 165, "duration_ms": 1234.56,
 "status": "success"}
```

In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  AILogger — Structured logging for AI interactions          ║
# ║                                                             ║
# ║  Logs EVERYTHING: timestamp, model, prompt preview,         ║
# ║  response preview, tokens, duration, chain step, batch ID,  ║
# ║  and status (success/error).                                ║
# ╚══════════════════════════════════════════════════════════════╝

class AILogger:
    """
    A structured logger for AI interactions.

    This is your 'black box' recorder — it captures every AI call
    so you can analyze, debug, and audit later.

    Key Design Decisions:
    - JSONL format: Append-only, easy to parse, works with big data tools
    - One file per run: Named with timestamp, never overwrites
    - Console + File: Real-time monitoring + permanent record
    """

    def __init__(self, log_dir: str = "logs"):
        """
        Initialize the logger.
        Creates a 'logs/' directory and opens a new log file named
        with the current timestamp.

        Args:
            log_dir: Directory to store log files (default: 'logs')
        """
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(parents=True, exist_ok=True)

        # Filename: logs/ai_log_2026-06-15_10-30-00.jsonl
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        self.log_file = self.log_dir / f"ai_log_{timestamp}.jsonl"

        # Standard Python logging for console output
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s | %(levelname)-8s | %(message)s",
            datefmt="%H:%M:%S",
        )
        self.console = logging.getLogger("AI-Script")

    def log_call(
        self,
        model: str,
        prompt: str,
        response_text: str,
        prompt_tokens: int = 0,
        completion_tokens: int = 0,
        total_tokens: int = 0,
        duration_ms: float = 0.0,
        chain_step: Optional[str] = None,
        batch_id: Optional[str] = None,
        status: str = "success",
        extra: Optional[dict] = None,
    ):
        """
        Log ONE AI interaction.

        This is called after EVERY AI API call, whether success or error.
        It writes a structured JSON object to the log file and prints
        a one-line summary to the console.

        Args:
            model: Model name (e.g., 'gpt-4o-mini')
            prompt: The input text sent to the AI
            response_text: The AI's response
            prompt_tokens: Number of input tokens
            completion_tokens: Number of output tokens
            total_tokens: Total tokens
            duration_ms: Time taken in milliseconds
            chain_step: Which step in a chain (e.g., 'summarize', 'translate')
            batch_id: Batch identifier for grouping related calls
            status: 'success' or 'error'
            extra: Any additional data to include
        """
        # ── Build structured log entry ─────────────────────
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "chain_step": chain_step,
            "batch_id": batch_id,
            "prompt_preview": prompt[:200] + ("..." if len(prompt) > 200 else ""),
            "response_preview": response_text[:200] + ("..." if len(response_text) > 200 else ""),
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
            "duration_ms": round(duration_ms, 2),
            "status": status,
        }
        if extra:
            log_entry.update(extra)

        # ── Write to JSONL file (append mode) ─────────────
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_entry) + "\n")

        # ── Console output (one line per call) ────────────
        tokens_str = f" [{prompt_tokens}in + {completion_tokens}out = {total_tokens}tok]"
        chain_str = f" [{chain_step}]" if chain_step else ""
        batch_str = f" [batch:{batch_id}]" if batch_id else ""
        duration_str = f" [{duration_ms:.0f}ms]"

        status_icon = "✓" if status == "success" else "✗"
        self.console.info(f"{status_icon} {chain_str}{batch_str}{tokens_str}{duration_str}")

    def summary(self) -> dict:
        """
        Read the current log file and produce a summary.

        Returns:
            dict with: total_calls, errors, total_tokens, avg_duration_ms, log_file
        """
        if not self.log_file.exists():
            return {"error": "No log file found"}

        calls = 0
        errors = 0
        total_tok = 0
        durations = []

        with open(self.log_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                entry = json.loads(line)
                calls += 1
                if entry.get("status") == "error":
                    errors += 1
                total_tok += entry.get("total_tokens", 0)
                durations.append(entry.get("duration_ms", 0))

        avg_duration = sum(durations) / len(durations) if durations else 0
        return {
            "log_file": str(self.log_file),
            "total_calls": calls,
            "errors": errors,
            "total_tokens": total_tok,
            "avg_duration_ms": round(avg_duration, 2),
        }

    def export_csv(self, csv_path: str = "logs/ai_log_export.csv"):
        """
        Export the JSONL log to CSV for spreadsheet analysis.

        This lets you open the log in Excel/Google Sheets
        for charting and analysis.
        """
        if not self.log_file.exists():
            print("No log file to export.")
            return

        with open(self.log_file, "r", encoding="utf-8") as f:
            entries = [json.loads(line) for line in f if line.strip()]

        if not entries:
            print("No log entries to export.")
            return

        fieldnames = list(entries[0].keys())
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(entries)
        print(f"📊 Exported {len(entries)} log entries to {csv_path}")


# Create a global logger instance — all functions share this.
ai_logger = AILogger()

print("✅ AILogger created — logs will be saved to: logs/ai_log_*.jsonl")

✅ AILogger created — logs will be saved to: logs/ai_log_*.jsonl


### 🧪 Quick Test: Try the Logger

Let's write a few sample log entries to see how it works.

In [6]:
# ── Test the logger ─────────────────────────────────────────
# These are FAKE entries just to show the logging format.

ai_logger.log_call(
    model="gpt-4o-mini",
    prompt="What is Python?",
    response_text="Python is a high-level programming language...",
    prompt_tokens=15,
    completion_tokens=42,
    total_tokens=57,
    duration_ms=850.0,
    chain_step="test",
    batch_id="demo-001",
)

# Log an error entry too
ai_logger.log_call(
    model="gpt-4o-mini",
    prompt="This will fail",
    response_text="ERROR: Rate limit exceeded",
    prompt_tokens=10,
    completion_tokens=0,
    total_tokens=10,
    duration_ms=5200.0,
    chain_step="test",
    batch_id="demo-001",
    status="error",
)

print("\n📋 Logger Summary:")
print(json.dumps(ai_logger.summary(), indent=2))

22:46:51 | INFO     | ✓  [test] [batch:demo-001] [15in + 42out = 57tok] s]
22:46:51 | INFO     | ✗  [test] [batch:demo-001] [10in + 0out = 10tok] [5200ms]



📋 Logger Summary:
{
  "log_file": "logs\\ai_log_2026-06-15_22-44-12.jsonl",
  "total_calls": 2,
  "errors": 1,
  "total_tokens": 67,
  "avg_duration_ms": 3025.0
}


In [7]:
# ── Let's peek at what was written to the log file ─────────
log_file = ai_logger.log_file
if log_file.exists():
    print(f"📄 Log file: {log_file}\n")
    with open(log_file, "r") as f:
        for i, line in enumerate(f, 1):
            entry = json.loads(line)
            print(f"Entry {i}:")
            print(f"  Time   : {entry['timestamp']}")
            print(f"  Status : {entry['status']}")
            print(f"  Tokens : {entry['total_tokens']}")
            print(f"  Duration: {entry['duration_ms']}ms")
            print()

📄 Log file: logs\ai_log_2026-06-15_22-44-12.jsonl

Entry 1:
  Time   : 2026-06-15T22:46:51.075024
  Status : success
  Tokens : 57
  Duration: 850.0ms

Entry 2:
  Time   : 2026-06-15T22:46:51.083619
  Status : error
  Tokens : 10
  Duration: 5200.0ms



### 📌 Key Takeaways — Logging

| Concept | Why It Matters |
|---------|----------------|
| **JSONL format** | Append-only, machine-readable, works with BigQuery/Splunk/ELK |
| **Structured fields** | token counts, durations, status — filterable and sortable |
| **One file per run** | Never lose history; compare runs side by side |
| **Console + File** | Real-time awareness + permanent record |
| **CSV export** | Business teams can open logs in Excel for cost analysis |

> **💡 Rule of thumb:** Every AI call in production MUST be logged. If you can't see what the AI did, you can't trust it.

---
---

# ⚡ PART 2: BATCH PROCESSING — Speed Up Your AI Workflows

## The Problem

```
You have 500 customer reviews to classify.
Sequential: 500 × 1.5s = 750s = 12.5 minutes 😴
Concurrent:  500 ÷ 10 at a time × 1.5s = ~75s = 1.25 minutes 🚀
```

## The Solution

Batch processing = sending MANY AI requests efficiently.

### Three Strategies We'll Cover

| Strategy | Speed | Complexity | Best For |
|----------|-------|------------|----------|
| **Sequential** | 🐢 Slow | 🟢 Simple | Small batches (<10), debugging |
| **Concurrent** | 🚀 Fast | 🟡 Medium | Large batches, independent items |
| **With Retry** | 🐢→🚀 | 🟠 Robust | Unreliable networks, rate limits |

## The Core AI Call Function

First, let's build the **fundamental building block** — a single AI call with logging built in.

In [8]:
# ─────────────────────────────────────────────────────────────
#  DEMO MODE: Simulated AI responses
# ─────────────────────────────────────────────────────────────
# When no API key is set, these simulate realistic responses
# so you can learn the patterns without spending money.

DEMO_RESPONSES = {
    "python": "Python is a versatile, high-level programming language known for its readability and extensive ecosystem of libraries. It's widely used in data science, web development, and AI.",
    "javascript": "JavaScript is the language of the web — it runs in every browser and enables interactive web pages. With Node.js, it also powers backend servers.",
    "rust": "Rust is a systems programming language focused on safety, speed, and concurrency. It prevents memory bugs at compile time without needing a garbage collector.",
    "tokyo": "Tokyo is the capital of Japan, known for its blend of ultramodern skyscrapers and traditional temples. It's one of the world's most populous cities.",
    "tokyo capital": "The capital of Japan is Tokyo. It has been the capital since 1868 when the imperial court moved from Kyoto.",
    "renewable": "Renewable energy comes from natural sources that replenish faster than they are consumed — like sunlight, wind, and water. It's key to fighting climate change.",
}

def _demo_ai_call(prompt: str, system_prompt: str = "") -> dict:
    """Simulate an AI call for demo mode."""
    start = time.time()

    # Try to match keywords in the prompt
    prompt_lower = prompt.lower()
    response = ""
    for keyword, demo_response in DEMO_RESPONSES.items():
        if keyword in prompt_lower:
            response = demo_response
            break

    if not response:
        # Generic fallback
        topics = ["artificial intelligence", "machine learning", "data science",
                  "cloud computing", "cybersecurity", "blockchain"]
        topic = random.choice(topics)
        response = f"{topic} is an exciting and rapidly evolving field that offers numerous opportunities for innovation and growth in the modern technology landscape."

    # Simulate realistic timing and token counts
    time.sleep(random.uniform(0.3, 0.8))
    prompt_tokens = len(prompt.split()) * 1.3
    completion_tokens = len(response.split()) * 1.4
    duration_ms = (time.time() - start) * 1000

    return {
        "text": response,
        "tokens": {
            "prompt": int(prompt_tokens),
            "completion": int(completion_tokens),
            "total": int(prompt_tokens + completion_tokens),
        },
        "duration_ms": duration_ms,
        "status": "success",
    }


print("✅ Demo AI responder ready — will match keywords and simulate responses")
print(f"   Registered keywords: {', '.join(DEMO_RESPONSES.keys())}")

✅ Demo AI responder ready — will match keywords and simulate responses
   Registered keywords: python, javascript, rust, tokyo, tokyo capital, renewable


In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  call_ai_sync — The fundamental AI call (with logging)      ║
# ║                                                             ║
# ║  Every AI pattern in this notebook builds on THIS function. ║
# ║  It handles:                                                ║
# ║    - Making the API call (or falling back to demo)          ║
# ║    - Extracting text and token counts                      ║
# ║    - Catching and logging errors                           ║
# ║    - Logging EVERYTHING to the AILogger                    ║
# ╚══════════════════════════════════════════════════════════════╝

def call_ai_sync(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    model: str = DEFAULT_MODEL,
    temperature: float = 0.3,
    max_tokens: int = 500,
    chain_step: Optional[str] = None,
    batch_id: Optional[str] = None,
) -> dict:
    """
    Make a SINGLE AI call and log it.

    This is the BASE FUNCTION for everything else in this notebook.
    Whether you're batch-processing, chaining, or building a project,
    it all starts here.

    What it does:
      1. Sends prompt + system_prompt to the AI model
      2. Captures the response text
      3. Records token usage and timing
      4. Logs everything via ai_logger.log_call()
      5. Returns a structured dict

    Returns:
      {
        "text": "...",          # The AI's response
        "tokens": {"prompt": N, "completion": N, "total": N},
        "duration_ms": 1234.56,
        "status": "success" | "error"
      }
    """
    start_time = time.time()
    response_text = ""
    prompt_tokens = 0
    completion_tokens = 0
    total_tokens = 0
    status = "success"

    try:
        if DEMO_MODE:
            # ── Demo mode: simulate ─────────────────────────────
            result = _demo_ai_call(prompt, system_prompt)
            response_text = result["text"]
            prompt_tokens = result["tokens"]["prompt"]
            completion_tokens = result["tokens"]["completion"]
            total_tokens = result["tokens"]["total"]
        else:
            # ── Real mode: call OpenAI API ──────────────────────
            completion = get_client().chat.completions.create(
                model=model,
                messages=[
                    {"role": "developer", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                temperature=temperature,
                max_completion_tokens=max_tokens,
            )
            response_text = completion.choices[0].message.content or ""
            usage = completion.usage
            if usage:
                prompt_tokens = usage.prompt_tokens
                completion_tokens = usage.completion_tokens
                total_tokens = usage.total_tokens

    except Exception as e:
        response_text = f"ERROR: {str(e)}"
        status = "error"

    duration_ms = (time.time() - start_time) * 1000

    # ── Log every call ───────────────────────────────────────────
    ai_logger.log_call(
        model=model,
        prompt=prompt,
        response_text=response_text,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=total_tokens,
        duration_ms=duration_ms,
        chain_step=chain_step,
        batch_id=batch_id,
        status=status,
    )

    return {
        "text": response_text,
        "tokens": {
            "prompt": prompt_tokens,
            "completion": completion_tokens,
            "total": total_tokens,
        },
        "duration_ms": duration_ms,
        "status": status,
    }


print("✅ call_ai_sync() defined — the foundation of all AI patterns")

✅ call_ai_sync() defined — the foundation of all AI patterns


### 🧪 Quick Test: Single AI Call

In [11]:
result = call_ai_sync(
    prompt="Explain what Python is in one sentence.",
    system_prompt="You are a teacher. Be concise.",
)

print(f"🤖 AI Response: {result['text']}")
print(f"📊 Tokens: {result['tokens']}")
print(f"⏱️  Duration: {result['duration_ms']:.0f}ms")

22:49:39 | INFO     | ✓  [9in + 36out = 45tok] s]


🤖 AI Response: Python is a versatile, high-level programming language known for its readability and extensive ecosystem of libraries. It's widely used in data science, web development, and AI.
📊 Tokens: {'prompt': 9, 'completion': 36, 'total': 45}
⏱️  Duration: 483ms


---
## 2.1 Sequential Batch Processing

The simplest approach: process items **one at a time** in a loop.

```python
for each item in my_list:
    result = call_ai(item)  # Wait for this to finish
    save(result)            # Then move to next
```

### When to Use
✅ Small batches (< 10 items)
✅ When order matters (step B depends on step A's result)
✅ Debugging and testing
✅ When you need simple, predictable behavior

### When NOT to Use
❌ Large batches (100+ items) — too slow
❌ When speed matters

In [12]:
def process_batch_sequential(
    prompts: List[str],
    system_prompt: str = "You are a helpful assistant.",
    model: str = DEFAULT_MODEL,
    temperature: float = 0.3,
    max_tokens: int = 500,
    batch_id: str = "seq-batch",
    verbose: bool = True,
) -> List[dict]:
    """
    Process a batch of prompts SEQUENTIALLY (one at a time).

    How it works:
      1. Loop through each prompt
      2. Call call_ai_sync() for each
      3. Wait for it to finish before starting the next
      4. Collect all results

    Time complexity: O(n) where n = number of prompts
    Wall-clock time: SUM of all individual call times

    Args:
        prompts: List of text prompts to send
        system_prompt: System instruction for all calls
        model: Model name
        temperature: Creativity (0 = deterministic, 1 = creative)
        max_tokens: Max response length
        batch_id: Label for grouping in logs
        verbose: Print progress

    Returns:
        List of result dicts (one per prompt)
    """
    if verbose:
        print(f"\n{'─' * 50}")
        print(f"  📦 SEQUENTIAL BATCH: {len(prompts)} prompts")
        print(f"{'─' * 50}")

    results = []
    for i, prompt in enumerate(prompts):
        if verbose:
            print(f"\n  ▶️  Item {i+1}/{len(prompts)}")
            print(f"      Prompt: {prompt[:60]}...")

        result = call_ai_sync(
            prompt=prompt,
            system_prompt=system_prompt,
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            batch_id=batch_id,
        )
        results.append(result)

        if verbose:
            print(f"      ✅ {result['status']} — {result['tokens']['total']} tokens")
            print(f"      Response: {result['text'][:80]}...")

    # ── Summary ─────────────────────────────────────────────
    total_tokens = sum(r["tokens"]["total"] for r in results)
    total_time = sum(r["duration_ms"] for r in results)
    errors = [r for r in results if r["status"] == "error"]

    if verbose:
        print(f"\n  {'─' * 30}")
        print(f"  ✅ SEQUENTIAL BATCH COMPLETE")
        print(f"     Items: {len(results)}")
        print(f"     Errors: {len(errors)}")
        print(f"     Total tokens: {total_tokens}")
        print(f"     Total time: {total_time/1000:.1f}s")
        print(f"     Avg time/item: {total_time/len(results)/1000:.2f}s")

    return results

In [13]:
# ── DEMO: Sequential Batch ─────────────────────────────────

batch_prompts = [
    "What is Python best used for?",
    "What is JavaScript best used for?",
    "What is Rust best used for?",
]

seq_results = process_batch_sequential(
    prompts=batch_prompts,
    system_prompt="Answer in 1-2 sentences.",
    batch_id="sequential-demo",
)

print("\n" + "═" * 50)
for i, r in enumerate(seq_results):
    print(f"  [{i+1}] {r['text'][:100]}")


──────────────────────────────────────────────────
  📦 SEQUENTIAL BATCH: 3 prompts
──────────────────────────────────────────────────

  ▶️  Item 1/3
      Prompt: What is Python best used for?...


22:50:06 | INFO     | ✓  [batch:sequential-demo] [7in + 36out = 44tok] s]


      ✅ success — 44 tokens
      Response: Python is a versatile, high-level programming language known for its readability...

  ▶️  Item 2/3
      Prompt: What is JavaScript best used for?...


22:50:07 | INFO     | ✓  [batch:sequential-demo] [7in + 35out = 42tok] s]


      ✅ success — 42 tokens
      Response: JavaScript is the language of the web — it runs in every browser and enables int...

  ▶️  Item 3/3
      Prompt: What is Rust best used for?...


22:50:07 | INFO     | ✓  [batch:sequential-demo] [7in + 33out = 41tok] s]


      ✅ success — 41 tokens
      Response: Rust is a systems programming language focused on safety, speed, and concurrency...

  ──────────────────────────────
  ✅ SEQUENTIAL BATCH COMPLETE
     Items: 3
     Errors: 0
     Total tokens: 127
     Total time: 1.8s
     Avg time/item: 0.61s

══════════════════════════════════════════════════
  [1] Python is a versatile, high-level programming language known for its readability and extensive ecosy
  [2] JavaScript is the language of the web — it runs in every browser and enables interactive web pages. 
  [3] Rust is a systems programming language focused on safety, speed, and concurrency. It prevents memory


---
## 2.2 Concurrent Batch Processing (Async)

This is where the **real speed** comes in! Instead of waiting for each call to finish, we send MULTIPLE requests at the same time.

### How Asyncio Works

```
Sequential:    ████████░░░░░░░░░░░░████████░░░░░░░░░░░░████████
               ▲ item 1 done ▲ item 2 done ▲ item 3 done

Concurrent:    ████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
               ████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
               ████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
               ▲ ALL 3 FINISH AT ABOUT THE SAME TIME!
```

### The Semaphore — Your Rate Limit Bouncer

```
        ┌─────────────────────────────────────┐
        │         SEMAPHORE (limit=5)          │
        │                                     │
        │  [Req1] [Req2] [Req3] [Req4] [Req5] │  ← 5 running
        │                                     │
        │         [Req6] [Req7] ...            │  ← Waiting
        └─────────────────────────────────────┘
```

The semaphore ensures you never exceed your API rate limit.

In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  call_ai_async — Async version for concurrent calls         ║
# ║                                                             ║
# ║  "async" = non-blocking. While waiting for the API,         ║
#  ║  Python can work on OTHER tasks (other API calls).         ║
# ║                                                             ║
# ║  "await" = "I'll wait here, but let other tasks run."      ║
# ╚══════════════════════════════════════════════════════════════╝

async def call_ai_async(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    model: str = DEFAULT_MODEL,
    temperature: float = 0.3,
    max_tokens: int = 500,
    chain_step: Optional[str] = None,
    batch_id: Optional[str] = None,
    semaphore: Optional[asyncio.Semaphore] = None,
) -> dict:
    """
    ASYNCHRONOUS AI call with optional semaphore for rate limiting.

    Same as call_ai_sync() but NON-BLOCKING.
    Use this when you want to run many calls concurrently.

    Args:
        semaphore: If provided, the call will acquire the semaphore
                   before making the API request. This limits how
                   many requests run simultaneously.
    """
    start_time = time.time()
    response_text = ""
    prompt_tokens = 0
    completion_tokens = 0
    total_tokens = 0
    status = "success"

    try:
        if DEMO_MODE:
            # Demo mode (simulated async delay)
            if semaphore:
                async with semaphore:
                    result = _demo_ai_call(prompt, system_prompt)
                    await asyncio.sleep(random.uniform(0.2, 0.4))
            else:
                result = _demo_ai_call(prompt, system_prompt)
                await asyncio.sleep(random.uniform(0.2, 0.4))

            response_text = result["text"]
            prompt_tokens = result["tokens"]["prompt"]
            completion_tokens = result["tokens"]["completion"]
            total_tokens = result["tokens"]["total"]
        else:
            # Real mode: actual async API call
            if semaphore:
                async with semaphore:
                    response = await get_async_client().chat.completions.create(
                        model=model,
                        messages=[
                            {"role": "developer", "content": system_prompt},
                            {"role": "user", "content": prompt},
                        ],
                        temperature=temperature,
                        max_completion_tokens=max_tokens,
                    )
            else:
                response = await get_async_client().chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "developer", "content": system_prompt},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=temperature,
                    max_completion_tokens=max_tokens,
                )

            response_text = response.choices[0].message.content or ""
            usage = response.usage
            if usage:
                prompt_tokens = usage.prompt_tokens
                completion_tokens = usage.completion_tokens
                total_tokens = usage.total_tokens

    except Exception as e:
        response_text = f"ERROR: {str(e)}"
        status = "error"

    duration_ms = (time.time() - start_time) * 1000

    # ── Log the call (same logger, same format) ───────────────
    ai_logger.log_call(
        model=model,
        prompt=prompt,
        response_text=response_text,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=total_tokens,
        duration_ms=duration_ms,
        chain_step=chain_step,
        batch_id=batch_id,
        status=status,
    )

    return {
        "text": response_text,
        "tokens": {
            "prompt": prompt_tokens,
            "completion": completion_tokens,
            "total": total_tokens,
        },
        "duration_ms": duration_ms,
        "status": status,
    }


print("✅ call_ai_async() defined — for concurrent batch processing")

✅ call_ai_async() defined — for concurrent batch processing


In [ ]:
async def process_batch_concurrent(
    prompts: List[str],
    system_prompt: str = "You are a helpful assistant.",
    model: str = DEFAULT_MODEL,
    temperature: float = 0.3,
    max_tokens: int = 500,
    batch_id: str = "async-batch",
    max_concurrency: int = 5,
    verbose: bool = True,
) -> List[dict]:
    """
    Process a batch CONCURRENTLY — many calls at the same time.

    How it works (step by step):
      1. Create an asyncio.Semaphore(max_concurrency)
         → This limits how many API calls run at once
      2. Build a list of coroutines (one per prompt)
         → Each coroutine is call_ai_async() with the semaphore
      3. Use asyncio.gather() to RUN ALL AT ONCE
         → They execute concurrently, sharing the semaphore
      4. Collect results as each finishes

    Time complexity: O(n) but wall-clock time is ~ (n / concurrency) × avg_time

    Args:
        prompts: List of text prompts
        max_concurrency: Max simultaneous API calls (default: 5)
                         Set based on your API tier's rate limit
        verbose: Print progress

    Returns:
        List of result dicts
    """
    if verbose:
        print(f"\n{'─' * 50}")
        print(f"  📦 CONCURRENT BATCH: {len(prompts)} prompts")
        print(f"      Max concurrency: {max_concurrency} at a time")
        print(f"{'─' * 50}")

    semaphore = asyncio.Semaphore(max_concurrency)

    # Build one task per prompt
    tasks = [
        call_ai_async(
            prompt=prompt,
            system_prompt=system_prompt,
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            batch_id=batch_id,
            semaphore=semaphore,
        )
        for prompt in prompts
    ]

    # Run all concurrently! asyncio.gather() starts ALL tasks
    # and waits for ALL to complete.
    # return_exceptions=True means one failure doesn't cancel others
    raw_results = await asyncio.gather(*tasks, return_exceptions=True)

    # Convert any exceptions into error dicts
    results = []
    for r in raw_results:
        if isinstance(r, Exception):
            results.append({
                "text": f"ERROR: {str(r)}",
                "tokens": {"prompt": 0, "completion": 0, "total": 0},
                "duration_ms": 0,
                "status": "error",
            })
        else:
            results.append(r)

    # ── Summary ─────────────────────────────────────────────
    total_tokens = sum(r["tokens"]["total"] for r in results)
    total_time = sum(r["duration_ms"] for r in results)
    errors = [r for r in results if r["status"] == "error"]

    if verbose:
        print(f"\n  {'─' * 30}")
        print(f"  ✅ CONCURRENT BATCH COMPLETE")
        print(f"     Items: {len(results)}")
        print(f"     Errors: {len(errors)}")
        print(f"     Total tokens: {total_tokens}")
        print(f"     Sum of durations: {total_time/1000:.1f}s")
        print(f"     (Actual wall-clock time is much less due to concurrency!)")

    return results

In [ ]:
# ── DEMO: Concurrent Batch ─────────────────────────────────

async def run_concurrent_demo():
    con_prompts = [
        "What is the capital of Japan?",
        "What is 2 + 2?",
        "What color is the sky?",
        "How many days in a week?",
        "What is water made of?",
    ]

    print("\n⏱️  Starting CONCURRENT batch (watch the speed!)\n")
    start_wall = time.time()

    con_results = await process_batch_concurrent(
        prompts=con_prompts,
        system_prompt="Answer very briefly.",
        batch_id="concurrent-demo",
        max_concurrency=5,
    )

    wall_time = time.time() - start_wall

    print(f"\n{'═' * 50}")
    print(f"  ⏱️  Actual wall-clock time: {wall_time:.2f}s")
    print(f"  (Sequential would take ~{sum(r['duration_ms'] for r in con_results)/1000:.1f}s)")
    print(f"  Speedup: {sum(r['duration_ms'] for r in con_results)/(wall_time*1000):.1f}x")
    print()

    for i, r in enumerate(con_results):
        print(f"  [{i+1}] {r['text'][:80]}")


await run_concurrent_demo()

---
## 2.3 Batch Processing with Retry Logic

### Why Retry?

API calls fail. It's a fact of life. Common failures:
- **429 Rate Limit** — too many requests too fast
- **5xx Server Errors** — temporary OpenAI issues
- **Network Timeouts** — internet hiccups

The solution: **Exponential Backoff**

```
Failed? → Wait 2s → Try again
Failed? → Wait 4s → Try again
Failed? → Wait 8s → Try again
Still failed? → Give up, log the error, move on
```

> **Key insight:** One bad item should NEVER crash your entire batch. Fail gracefully.

In [ ]:
def process_batch_with_retry(
    prompts: List[str],
    system_prompt: str = "You are a helpful assistant.",
    model: str = DEFAULT_MODEL,
    max_retries: int = 3,
    batch_id: str = "retry-batch",
    verbose: bool = True,
) -> List[dict]:
    """
    Process a batch with automatic retry on failure.

    Strategy:
      1. Try the API call
      2. If it fails → wait (exponential backoff: 2^attempt seconds)
      3. Try again (up to max_retries times)
      4. If all attempts fail → mark as error and continue

    The exponential backoff looks like:
      Attempt 1 fail → wait 2s
      Attempt 2 fail → wait 4s
      Attempt 3 fail → wait 8s
      Attempt 4 fail → give up

    This pattern is called "Retry with Exponential Backoff" and is
    the industry standard for resilient API integrations.

    Args:
        prompts: List of prompts to process
        max_retries: Maximum number of retry attempts per item
        verbose: Print progress
    """
    if verbose:
        print(f"\n{'─' * 50}")
        print(f"  📦 BATCH WITH RETRY: {len(prompts)} prompts")
        print(f"      Max retries per item: {max_retries}")
        print(f"      Backoff: exponential (2^attempt seconds)")
        print(f"{'─' * 50}")

    results = []
    for i, prompt in enumerate(prompts):
        if verbose:
            print(f"\n  ▶️  Item {i+1}/{len(prompts)}")

        attempt = 0
        success = False
        result = None

        while attempt <= max_retries and not success:
            if attempt > 0:
                # Exponential backoff: wait 2^attempt seconds
                wait_time = 2 ** attempt
                if verbose:
                    print(f"      🔄 Retry {attempt}/{max_retries} after {wait_time}s...")
                time.sleep(wait_time)

            result = call_ai_sync(
                prompt=prompt,
                system_prompt=system_prompt,
                model=model,
                batch_id=batch_id,
            )
            attempt += 1

            if result["status"] == "success":
                success = True
                if verbose:
                    print(f"      ✅ Succeeded on attempt {attempt}")
            elif attempt <= max_retries:
                if verbose:
                    print(f"      ❌ Failed ({result['text'][:50]}...)")

        results.append(result)

    # ── Summary ─────────────────────────────────────────────
    total_tokens = sum(r["tokens"]["total"] for r in results)
    errors = [r for r in results if r["status"] == "error"]

    if verbose:
        print(f"\n  {'─' * 30}")
        print(f"  ✅ RETRY BATCH COMPLETE")
        print(f"     Items: {len(results)}")
        print(f"     Errors: {len(errors)}")
        print(f"     Total tokens: {total_tokens}")

    return results

In [ ]:
# ── DEMO: Batch with Retry ─────────────────────────────────
# (In demo mode, calls won't actually fail, but you can see
#  the retry pattern structure)

retry_results = process_batch_with_retry(
    prompts=[
        "List 2 benefits of Python.",
        "List 2 benefits of TypeScript.",
    ],
    batch_id="retry-demo",
)

print("\n" + "═" * 50)
for i, r in enumerate(retry_results):
    print(f"  [{i+1}] {r['text'][:100]}")

### 📌 Key Takeaways — Batch Processing

| Concept | Why It Matters |
|---------|----------------|
| **Concurrency** | 5-10x speedup for independent API calls |
| **Semaphore** | Prevents rate limit errors (your "bouncer") |
| **Exponential Backoff** | Graceful handling of transient failures |
| **Error Isolation** | One failure doesn't crash the whole batch |
| **Token Tracking** | Know the cost of each batch immediately |

> **💡 Choosing the right method:**
> - < 10 items → Sequential (simple and predictable)
> - 10-100 items → Concurrent with Semaphore
> - > 100 items → Concurrent + Retry + Progress logging
> - Unreliable network → Always use retry

---
---

# 🔗 PART 3: CHAINING — AI Assembly Lines

## What is Chaining?

Chaining = the **output of one AI call becomes the input to the next**. Think of it as an assembly line for information.

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  STEP 1  │───▶│  STEP 2  │───▶│  STEP 3  │───▶│  STEP 4  │
│ Outline  │    │  Write   │    │Summarize │    │Translate │
└──────────┘    └──────────┘    └──────────┘    └──────────┘
     │              │              │              │
     ▼              ▼              ▼              ▼
  "Topic: AI"   "Article..."   "TL;DR..."    "Resumen..."
```

## Why Chain?

| Benefit | Explanation |
|---------|-------------|
| 🧩 **Break complexity** | Divide a hard task into simpler sub-tasks |
| 🔍 **Debug clarity** | Each step has ONE job — easy to find where things go wrong |
| 💰 **Cost optimization** | Use cheap models for simple steps, expensive ones for complex |
| 🎯 **Different temperatures** | Creative (T=0.8) for writing, factual (T=0.1) for extraction |
| 📋 **Full traceability** | Every step logged separately with its own chain_step label |

## Two Types of Chains

1. **Transform Chain** — Each step transforms the data (e.g., outline → article → summary)
2. **Augment Chain** — Each step adds information (e.g., classify + extract + sentiment)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  chain_step — A single step in a processing chain           ║
# ║                                                             ║
# ║  A thin wrapper around call_ai_sync() that just adds the    ║
# ║  step name for logging. Each step is tracked separately     ║
# ║  in the log with its chain_step label.                      ║
# ╚══════════════════════════════════════════════════════════════╝

def chain_step(
    prompt: str,
    system_prompt: str,
    step_name: str,
    **kwargs,
) -> dict:
    """
    Execute ONE step in an AI processing chain.

    This is a wrapper around call_ai_sync() that:
      1. Passes prompt + system_prompt to call_ai_sync
      2. Labels the call with chain_step=step_name
      3. Passes any extra **kwargs (model, temperature, etc.)

    The step name will appear in the logs, letting you trace
    exactly which step produced which output.

    Args:
        prompt: The input for THIS step (often the output of previous step)
        system_prompt: Specialized instruction for THIS step
        step_name: Label for logging (e.g., 'outline', 'write', 'translate')
        **kwargs: Any extra args passed to call_ai_sync (model, temp, etc.)

    Returns:
        Result dict from call_ai_sync
    """
    return call_ai_sync(
        prompt=prompt,
        system_prompt=system_prompt,
        chain_step=step_name,
        **kwargs,
    )


print("✅ chain_step() defined — building block for all chains")

---
## Chain Example 1: Content Generation Pipeline

A **4-step transform chain** that creates a complete content piece:

| Step | Name | Input → Output | Temperature |
|------|------|----------------|-------------|
| 1 | **Outline** | Topic → Structured outline | 0.5 (balanced) |
| 2 | **Write** | Outline → Full article | 0.7 (creative) |
| 3 | **Summarize** | Article → TL;DR | 0.2 (factual) |
| 4 | **Translate** | Summary → Target language | 0.1 (precise) |

> **Notice:** Each step has a DIFFERENT temperature! Creative writing (0.7) needs more "imagination" than translation (0.1).

In [ ]:
def run_content_pipeline(topic: str, language: str = "Spanish") -> dict:
    """
    A 4-STEP CONTENT GENERATION CHAIN.

    Each step depends on the output of the previous step:
      1. OUTLINE   → Generate a structured article outline
      2. WRITE     → Write a full article from the outline
      3. SUMMARIZE → Create a concise TL;DR summary
      4. TRANSLATE → Translate the summary to the target language

    This is a TRANSFORM chain — each step transforms the data.
    The final output contains ALL intermediate results for full traceability.

    Args:
        topic: The article topic
        language: Target language for translation

    Returns:
        dict with: topic, outline, article, summary, translation, tokens, time_ms
    """
    print(f"\n{'═' * 50}")
    print(f"  🔗 CONTENT PIPELINE (4-Step Chain)")
    print(f"  Topic: {topic}  →  Language: {language}")
    print(f"{'═' * 50}")

    # ── STEP 1: Generate outline ─────────────────────────────
    print(f"\n  📝 STEP 1/4: OUTLINE")
    step1 = chain_step(
        prompt=f"Create a detailed article outline for: {topic}",
        system_prompt=(
            "You are an expert content strategist. Create a detailed, "
            "well-structured outline with 4-5 sections and bullet points "
            "under each section."
        ),
        step_name="outline",
        temperature=0.5,
    )
    outline = step1["text"]
    if step1["status"] == "error":
        return {"error": f"Outline step failed: {outline}"}
    print(f"      ✅ Outline generated ({len(outline)} chars)")

    # ── STEP 2: Write article ────────────────────────────────
    print(f"\n  ✍️  STEP 2/4: WRITE ARTICLE")
    step2 = chain_step(
        prompt=f"Write a complete, engaging article based on this outline:\n\n{outline}",
        system_prompt=(
            "You are an experienced writer. Write engaging, well-researched "
            "content. Use examples, keep paragraphs concise, and maintain "
            "a professional but accessible tone."
        ),
        step_name="write",
        temperature=0.7,  # Higher temp for creative writing
        max_tokens=2000,
    )
    article = step2["text"]
    if step2["status"] == "error":
        return {"error": f"Write step failed: {article}"}
    print(f"      ✅ Article written ({len(article)} chars)")

    # ── STEP 3: Summarize ────────────────────────────────────
    print(f"\n  📋 STEP 3/4: SUMMARIZE")
    step3 = chain_step(
        prompt=f"Summarize the following article in 3-4 concise sentences. "
               f"Capture ONLY the key points:\n\n{article}",
        system_prompt=(
            "You are a concise summarizer. Capture the key points only. "
            "No fluff, no commentary, no introductory phrases."
        ),
        step_name="summarize",
        temperature=0.2,  # Low temp for factual summary
    )
    summary = step3["text"]
    print(f"      ✅ Summary created ({len(summary)} chars)")
    print(f"      Preview: {summary[:100]}...")

    # ── STEP 4: Translate ────────────────────────────────────
    print(f"\n  🌐 STEP 4/4: TRANSLATE TO {language.upper()}")
    step4 = chain_step(
        prompt=f"Translate the following summary to {language}. "
               f"Keep it natural and accurate:\n\n{summary}",
        system_prompt=(
            f"You are a professional translator. Translate to {language} "
            f"accurately and naturally. Preserve the meaning, not the word order."
        ),
        step_name="translate",
        temperature=0.1,  # Very low temp for translation accuracy
    )
    translation = step4["text"]
    print(f"      ✅ Translated to {language}")
    print(f"      Preview: {translation[:100]}...")

    # ── Chain Complete ───────────────────────────────────────
    total_tokens = (
        step1["tokens"]["total"]
        + step2["tokens"]["total"]
        + step3["tokens"]["total"]
        + step4["tokens"]["total"]
    )
    total_time = (
        step1["duration_ms"]
        + step2["duration_ms"]
        + step3["duration_ms"]
        + step4["duration_ms"]
    )

    print(f"\n{'═' * 50}")
    print(f"  ✅ CHAIN COMPLETE")
    print(f"     Total tokens: {total_tokens}")
    print(f"     Total time: {total_time/1000:.1f}s")
    print(f"     Avg tokens/step: {total_tokens // 4}")

    return {
        "topic": topic,
        "outline": outline,
        "article": article,
        "summary": summary,
        "translation": translation,
        "tokens": total_tokens,
        "time_ms": total_time,
    }


print("✅ run_content_pipeline() defined — 4-step AI content factory")

In [ ]:
# ── DEMO: Content Pipeline ─────────────────────────────────

pipeline_result = run_content_pipeline(
    topic="Benefits of Renewable Energy",
    language="French",
)

if "error" not in pipeline_result:
    print("\n" + "═" * 60)
    print("  📄 FULL PIPELINE OUTPUT")
    print("═" * 60)

    print("\n─── OUTLINE ───")
    print(pipeline_result["outline"][:400])

    print("\n─── SUMMARY ───")
    print(pipeline_result["summary"])

    print("\n─── TRANSLATION (French) ───")
    print(pipeline_result["translation"])

    print(f"\n📊 Total tokens used: {pipeline_result['tokens']}")
else:
    print(f"❌ Pipeline failed: {pipeline_result['error']}")

---
## Chain Example 2: Text Analysis Pipeline

An **AUGMENT chain** — each step ADDS new information:

| Step | Name | What It Does | Temp |
|------|------|-------------|------|
| 1 | **Classify** | Categorizes the text (complaint, question, feedback, etc.) | 0.0 |
| 2 | **Extract** | Pulls out entities (people, places, dates, order IDs) | 0.0 |
| 3 | **Sentiment** | Rates emotional tone (-10 to +10) with explanation | 0.2 |
| 4 | **Recommend** | Suggests next action based on all previous analysis | 0.3 |

> **Real-world use:** Customer support ticket triage, content moderation, document processing

In [ ]:
def run_analysis_chain(text: str) -> dict:
    """
    A 4-STEP TEXT ANALYSIS CHAIN.

    This is an AUGMENT chain — each step ADDS information:
      1. CLASSIFY  → Category (complaint, question, feedback, praise, spam)
      2. EXTRACT   → Named entities (people, orgs, dates, order IDs)
      3. SENTIMENT → Emotional tone rating + explanation
      4. RECOMMEND → Suggested next action based on full analysis

    Unlike the content pipeline, steps 1-3 operate on the ORIGINAL text
    and step 4 combines their outputs.

    Args:
        text: The text to analyze

    Returns:
        dict with: category, entities, sentiment, recommendation, tokens
    """
    print(f"\n{'═' * 50}")
    print(f"  🔗 ANALYSIS CHAIN (4-Step Augment Chain)")
    print(f"  Input: {text[:60]}...")
    print(f"{'═' * 50}")

    # ── Step 1: Classify ────────────────────────────────────
    print(f"\n  🏷️  STEP 1/4: CLASSIFY")
    step1 = chain_step(
        prompt=(
            f"Classify this text into EXACTLY ONE category: "
            f"complaint, question, feedback, praise, or spam.\n\n{text}"
        ),
        system_prompt=(
            "You are a text classifier. Reply with ONLY the category word, "
            "nothing else. No punctuation, no explanation."
        ),
        step_name="classify",
        temperature=0.0,  # Deterministic — no creativity needed
    )
    category = step1["text"].strip().lower()
    print(f"      ✅ Classified as: {category}")

    # ── Step 2: Extract entities ─────────────────────────────
    print(f"\n  🔍 STEP 2/4: EXTRACT ENTITIES")
    step2 = chain_step(
        prompt=(
            f"Extract ALL named entities from this text (people, "
            f"organizations, dates, locations, order IDs, product names). "
            f"Return as a comma-separated list.\n\n{text}"
        ),
        system_prompt=(
            "You are an entity extractor. Return ONLY the comma-separated "
            "list. If no entities found, return 'None'."
        ),
        step_name="extract",
        temperature=0.0,
    )
    entities = step2["text"]
    print(f"      ✅ Entities: {entities[:100]}...")

    # ── Step 3: Sentiment analysis ───────────────────────────
    print(f"\n  💭 STEP 3/4: SENTIMENT ANALYSIS")
    step3 = chain_step(
        prompt=(
            f"Analyze the sentiment of this text. Rate from -10 "
            f"(very negative) to +10 (very positive) and explain "
            f"your reasoning in one sentence.\n\n{text}"
        ),
        system_prompt=(
            "You are a sentiment analyst. Format your response as: "
            "'Score: N/10. Reason: ...' Be objective."
        ),
        step_name="sentiment",
        temperature=0.2,
    )
    sentiment = step3["text"]
    print(f"      ✅ Sentiment: {sentiment[:80]}...")

    # ── Step 4: Recommend action ─────────────────────────────
    print(f"\n  💡 STEP 4/4: RECOMMEND ACTION")
    step4 = chain_step(
        prompt=(
            f"Based on this full analysis:\n"
            f"Category: {category}\n"
            f"Entities: {entities}\n"
            f"Sentiment: {sentiment}\n\n"
            f"Original text: {text}\n\n"
            f"What is the BEST next action? Be specific and actionable."
        ),
        system_prompt=(
            "You are a customer service advisor. Recommend 1-2 specific, "
            "actionable next steps. Consider priority level and urgency."
        ),
        step_name="recommend",
        temperature=0.3,
    )
    recommendation = step4["text"]
    print(f"      ✅ Recommendation generated")

    # ── Chain Complete ───────────────────────────────────────
    total_tokens = (
        step1["tokens"]["total"]
        + step2["tokens"]["total"]
        + step3["tokens"]["total"]
        + step4["tokens"]["total"]
    )

    print(f"\n{'═' * 50}")
    print(f"  ✅ ANALYSIS CHAIN COMPLETE — Total tokens: {total_tokens}")

    return {
        "category": category,
        "entities": entities,
        "sentiment": sentiment,
        "recommendation": recommendation,
        "tokens": total_tokens,
    }


print("✅ run_analysis_chain() defined — 4-step text analysis pipeline")

In [ ]:
# ── DEMO: Analysis Chain ───────────────────────────────────

sample_text = (
    "I ordered your ProWidget X100 last week and it arrived damaged. "
    "The box was crushed and the screen is cracked. "
    "I need a replacement urgently. My order number is ORD-98765."
)

analysis = run_analysis_chain(sample_text)

if "error" not in analysis:
    print("\n" + "═" * 60)
    print("  📋 FULL ANALYSIS OUTPUT")
    print("═" * 60)
    print(f"\n  🏷️  Category:       {analysis['category']}")
    print(f"  🔍 Entities:       {analysis['entities']}")
    print(f"  💭 Sentiment:      {analysis['sentiment']}")
    print(f"  💡 Recommendation: {analysis['recommendation'][:200]}")
    print(f"\n  📊 Tokens used: {analysis['tokens']}")

### 📌 Key Takeaways — Chaining

| Concept | Why It Matters |
|---------|----------------|
| **Single Responsibility** | Each step does ONE thing → easy to debug and improve |
| **Temperature Tuning** | Each step can have its own creativity level |
| **Model Choice per Step** | Use cheap models for easy steps, expensive ones for complex |
| **Intermediate Results** | Save all step outputs — not just the final one |
| **Chain Types** | Transform (data changes) vs Augment (data grows) |

> **💡 When to chain:**
> - The task has CLEAR sub-tasks (outline → write → summarize)
> - Different sub-tasks need DIFFERENT capabilities
> - You need AUDITABILITY at each stage
> - A single prompt would be too long or complex

---
---

# 🏆 FINAL PROJECT: Multi-Topic Content Engine

## Combining Batch + Chain + Logging

Now let's put EVERYTHING together into a real-world project!

### The Scenario

You're an AI Engineer at a content marketing agency. Your boss says:

> *"We need blog content for 5 different topics, in 3 different languages each. Can you handle that?"*

Your solution:

1. **BATCH** — Process all 5 topics concurrently
2. **CHAIN** — For EACH topic, run the 4-step content pipeline
3. **LOG** — Every single AI call is logged for cost analysis

```
                    ┌───────────────────────┐
                    │    TOPIC BATCH         │
                    │  (5 topics at once)    │
                    └───────┬───────┬───────┘
                            │       │
              ┌─────────────┘       └─────────────┐
              ▼                                    ▼
    ┌──────────────────┐              ┌──────────────────┐
    │  Topic 1 Chain   │              │  Topic 5 Chain   │
    │  ├─ Outline      │              │  ├─ Outline      │
    │  ├─ Write        │    ...       │  ├─ Write        │
    │  ├─ Summarize    │              │  ├─ Summarize    │
    │  └─ Translate    │              │  └─ Translate    │
    └──────┬───────────┘              └──────┬───────────┘
           ▼                                 ▼
    ┌──────────────────┐              ┌──────────────────┐
    │  Logged to JSONL │              │  Logged to JSONL │
    └──────────────────┘              └──────────────────┘

Total AI calls: 5 topics × 4 steps = 20 calls
All logged, all tracked, all auditable!
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  FINAL PROJECT: Multi-Topic Content Engine                 ║
# ║                                                             ║
# ║  This is a PRODUCTION-GRADE script that combines:           ║
# ║    ✅ BATCH — Process multiple topics concurrently          ║
# ║    ✅ CHAIN — 4-step pipeline per topic (outline→write→    ║
# ║             summarize→translate)                            ║
# ║    ✅ LOG   — Every call tracked with tokens, timing,       ║
# ║             and status                                      ║
# ║                                                             ║
# ║  Bonus features:                                            ║
# ║    - Progress bar (visual feedback)                         ║
# ║    - Cost estimation (track spend per topic)                ║
# ║    - Summary report at the end                              ║
# ║    - CSV export for spreadsheet analysis                    ║
# ╚══════════════════════════════════════════════════════════════╝

import asyncio
from datetime import datetime


# ── Cost estimation (approximate, in USD) ───────────────────-
# These are approximate rates for gpt-4o-mini (as of 2026)
COST_PER_INPUT_TOKEN = 0.15 / 1_000_000    # $0.15 per 1M input tokens
COST_PER_OUTPUT_TOKEN = 0.60 / 1_000_000   # $0.60 per 1M output tokens


async def process_topic_async(
    topic: str,
    language: str,
    topic_id: int,
    semaphore: asyncio.Semaphore,
) -> dict:
    """
    Process ONE topic through the full pipeline (async).

    Each topic goes through: outline → write → summarize → translate
    All calls use the shared semaphore for rate limiting.
    """
    result = {
        "topic": topic,
        "language": language,
        "status": "pending",
        "steps": {},
        "total_tokens": 0,
        "total_cost": 0.0,
        "total_time_ms": 0,
    }

    try:
        # Step 1: Outline
        s1 = await call_ai_async(
            prompt=f"Create a detailed article outline for: {topic}",
            system_prompt="You are an expert content strategist. Create a "
                          "detailed, well-structured outline with sections "
                          "and bullet points.",
            chain_step="outline",
            batch_id=f"topic-{topic_id}",
            semaphore=semaphore,
            temperature=0.5,
        )
        result["steps"]["outline"] = s1
        if s1["status"] == "error":
            result["status"] = f"failed at outline: {s1['text'][:100]}"
            return result

        # Step 2: Write article
        s2 = await call_ai_async(
            prompt=f"Write a complete article based on this outline:\n\n{s1['text']}",
            system_prompt="You are an experienced writer. Write engaging, "
                          "well-structured content. Keep it professional.",
            chain_step="write",
            batch_id=f"topic-{topic_id}",
            semaphore=semaphore,
            temperature=0.7,
            max_tokens=2000,
        )
        result["steps"]["write"] = s2
        if s2["status"] == "error":
            result["status"] = f"failed at write: {s2['text'][:100]}"
            return result

        # Step 3: Summarize
        s3 = await call_ai_async(
            prompt=f"Summarize this article in 3-4 sentences:\n\n{s2['text']}",
            system_prompt="You are a concise summarizer. Key points only.",
            chain_step="summarize",
            batch_id=f"topic-{topic_id}",
            semaphore=semaphore,
            temperature=0.2,
        )
        result["steps"]["summarize"] = s3

        # Step 4: Translate
        s4 = await call_ai_async(
            prompt=f"Translate this summary to {language}:\n\n{s3['text']}",
            system_prompt=f"You are a professional translator for {language}.",
            chain_step="translate",
            batch_id=f"topic-{topic_id}",
            semaphore=semaphore,
            temperature=0.1,
        )
        result["steps"]["translate"] = s4

        # ── Calculate totals ─────────────────────────────────
        for step_name, step_result in result["steps"].items():
            result["total_tokens"] += step_result["tokens"]["total"]
            result["total_time_ms"] += step_result["duration_ms"]

        # Estimate cost
        total_input = sum(
            s["tokens"]["prompt"] for s in result["steps"].values()
        )
        total_output = sum(
            s["tokens"]["completion"] for s in result["steps"].values()
        )
        result["total_cost"] = round(
            total_input * COST_PER_INPUT_TOKEN
            + total_output * COST_PER_OUTPUT_TOKEN,
            6,
        )

        result["status"] = "success"

    except Exception as e:
        result["status"] = f"error: {str(e)}"

    return result


async def run_multi_topic_engine(
    topics: List[str],
    languages: List[str],
    max_concurrency: int = 5,
):
    """
    Full pipeline: Batch × Chain × Log

    For EACH topic, for EACH language, run the 4-step content chain.
    All topics are processed CONCURRENTLY.

    Args:
        topics: List of article topics
        languages: List of target languages for translation
        max_concurrency: Max simultaneous AI calls

    Returns:
        dict with: results (per topic), total_cost, total_tokens,
                   total_time, summary report
    """
    print("\n" + "═" * 60)
    print("  🏆 MULTI-TOPIC CONTENT ENGINE")
    print(f"  Topics: {len(topics)}  |  Languages: {len(languages)}")
    print(f"  Total pipelines: {len(topics) * len(languages)}")
    print(f"  Total AI calls: {len(topics) * len(languages) * 4}")
    print(f"  Max concurrency: {max_concurrency}")
    print("═" * 60)

    semaphore = asyncio.Semaphore(max_concurrency)
    start_time = time.time()

    # Build all tasks (topic × language combinations)
    all_tasks = []
    for i, topic in enumerate(topics):
        for lang in languages:
            task = process_topic_async(
                topic=topic,
                language=lang,
                topic_id=len(all_tasks) + 1,
                semaphore=semaphore,
            )
            all_tasks.append(task)

    # ── Run all concurrently ──────────────────────────────────
    print(f"\n  🚀 Launching {len(all_tasks)} pipelines concurrently...\n")
    topic_results = await asyncio.gather(*all_tasks, return_exceptions=True)

    # ── Process results ───────────────────────────────────────
    processed = []
    for r in topic_results:
        if isinstance(r, Exception):
            processed.append({
                "topic": "UNKNOWN",
                "language": "UNKNOWN",
                "status": f"fatal: {str(r)}",
                "total_tokens": 0,
                "total_cost": 0.0,
                "total_time_ms": 0,
                "steps": {},
            })
        else:
            processed.append(r)

    wall_time = time.time() - start_time

    # ── Generate Summary Report ───────────────────────────────
    total_tokens = sum(r["total_tokens"] for r in processed)
    total_cost = sum(r["total_cost"] for r in processed)
    success_count = sum(1 for r in processed if r["status"] == "success")
    failed = [r for r in processed if r["status"] != "success"]

    print(f"\n{'═' * 60}")
    print(f"  📊 FINAL REPORT")
    print(f"{'═' * 60}")
    print(f"  Pipelines completed: {success_count}/{len(processed)}")
    print(f"  Failed: {len(failed)}")
    print(f"  Total tokens used: {total_tokens:,}")
    print(f"  Estimated cost: ${total_cost:.4f}")
    print(f"  Wall-clock time: {wall_time:.2f}s")
    print(f"  Average per pipeline: {wall_time / len(processed):.2f}s")

    if failed:
        print(f"\n  ❌ Failed pipelines:")
        for f in failed:
            print(f"     - {f['topic']} ({f['language']}): {f['status']}")

    # ── Log file summary ───────────────────────────────────────
    log_summary = ai_logger.summary()
    print(f"\n  📋 Log file: {log_summary.get('log_file', 'N/A')}")
    print(f"  📋 Total API calls logged: {log_summary.get('total_calls', 0)}")

    # Export logs to CSV
    ai_logger.export_csv()

    return {
        "results": processed,
        "total_tokens": total_tokens,
        "total_cost": total_cost,
        "wall_time_s": round(wall_time, 2),
        "success_count": success_count,
        "failed_count": len(failed),
        "failed_details": failed,
    }


print("✅ Multi-Topic Content Engine defined — ready to run!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🚀 RUN THE FINAL PROJECT                                   ║
# ║                                                             ║
# ║  This executes the Multi-Topic Content Engine with:         ║
# ║   - 5 topics                                                ║
# ║   - 2 languages each                                        ║
# ║   - Concurrent execution (3 at a time)                      ║
# ║   - Full logging + CSV export                              ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Configuration ───────────────────────────────────────────
TOPICS = [
    "Benefits of Remote Work",
    "Introduction to Quantum Computing",
    "Sustainable Living Tips",
    "The Rise of Electric Vehicles",
    "Mental Health in the Digital Age",
]

LANGUAGES = ["Spanish", "German"]

print(f"🎯 Topics ({len(TOPICS)}):")
for t in TOPICS:
    print(f"    • {t}")
print(f"\n🌐 Languages: {', '.join(LANGUAGES)}")
print(f"⚙️  Max concurrency: 3")

# ── Run the engine ──────────────────────────────────────────
report = await run_multi_topic_engine(
    topics=TOPICS,
    languages=LANGUAGES,
    max_concurrency=3,
)

# ── Sample Outputs ──────────────────────────────────────────
print("\n" + "═" * 60)
print("  🎉 SAMPLE OUTPUTS")
print("═" * 60)

for r in report["results"][:2]:  # Show first 2 results
    if r["status"] == "success":
        print(f"\n  📄 Topic: {r['topic']} ({r['language']})")
        print(f"     Tokens: {r['total_tokens']}  |  Cost: ${r['total_cost']:.6f}")
        print(f"     Summary: {r['steps'].get('summarize', {}).get('text', 'N/A')[:120]}...")
        print(f"     Translation: {r['steps'].get('translate', {}).get('text', 'N/A')[:100]}...")

### 📊 Project Review

Let's peek at what the engine generated and the logs it created.

In [ ]:
# ── Show log summary ───────────────────────────────────────
print("📋 LOG SUMMARY")
print(json.dumps(ai_logger.summary(), indent=2))

# ── Show CSV export ────────────────────────────────────────
csv_path = "logs/ai_log_export.csv"
if os.path.exists(csv_path):
    with open(csv_path, "r") as f:
        lines = f.readlines()
    print(f"\n📊 CSV Export ({len(lines)-1} entries):")
    print(f"    File: {csv_path}")
    print(f"    Columns: {lines[0].strip()}")

# ── Show final report ──────────────────────────────────────
print(f"\n{'═' * 50}")
print(f"  🏆 PROJECT COMPLETE")
print(f"{'═' * 50}")
print(f"  ✅ {report['success_count']} pipelines succeeded")
print(f"  ❌ {report['failed_count']} pipelines failed")
print(f"  💰 Total cost: ${report['total_cost']:.4f}")
print(f"  ⏱️  Total time: {report['wall_time_s']}s")

---
---

# 🎓 Conclusion & Next Steps

## What You've Learned

| Pattern | Skill | Key Takeaway |
|---------|-------|--------------|
| **Logging** | Observability | Every AI call must be logged — know your tokens, cost, and errors |
| **Batch Processing** | Performance | Async concurrency gives 5-10x speedup; semaphores prevent rate limits |
| **Chaining** | Architecture | Break complex tasks into single-purpose steps with different settings |

## The Mental Model

```
When building an AI script, ALWAYS ask:

1. LOG:   "Can I see what every call did?"
2. BATCH: "Am I processing items efficiently?"
3. CHAIN: "Is there a clear pipeline or is it all one big prompt?"
```

## Next Steps for Your AI Engineering Journey

1. **Add Caching** — Cache identical prompts to save money (use `functools.lru_cache` or Redis)
2. **Add Validation** — Validate AI outputs with Pydantic schemas before using them
3. **Add Monitoring** — Send logs to a monitoring service (Datadog, Grafana, or even a simple dashboard)
4. **Add A/B Testing** — Compare different models/prompts on the same input
5. **Add Human-in-the-Loop** — Flag low-confidence results for human review

## Resources

- [OpenAI API Docs](https://platform.openai.com/docs)
- [Python Asyncio Guide](https://docs.python.org/3/library/asyncio.html)
- [Python Logging Cookbook](https://docs.python.org/3/howto/logging-cookbook.html)

---

*Happy building! 🚀*